# Clasificación de tumores de mama con SVM

## Definición del problema

**Pregunta de investigación:**
¿Es posible predecir si un tumor de mama es **maligno** o **benigno** a partir de
las características morfológicas de sus núcleos celulares?

**Unidad de análisis:**
Cada fila representa una **muestra de tumor de mama** (un caso individual),
caracterizada por 30 variables numéricas calculadas a partir de imágenes de núcleos
celulares (radio, textura, perímetro, área, suavidad, compacidad, concavidad, etc.).

**Variable objetivo (target):**
`target` — variable binaria que indica el diagnóstico:
`0 = maligno`, `1 = benigno`.

---

> ⚠️ **Advertencia:** Este ejercicio tiene fines exclusivamente **académicos y
> demostrativos**. Los resultados **no constituyen un diagnóstico médico** ni deben
> utilizarse para tomar decisiones clínicas. Cualquier interpretación en un contexto
> real requiere la evaluación de un profesional de la salud calificado.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

bunch = load_breast_cancer(as_frame=True)
X = bunch.data.copy()
y = bunch.target.copy()
print(X.shape)
print(y.value_counts().sort_index())
X.head()

(569, 30)
target
0    212
1    357
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### Comentario de la salida

La ejecución confirma la estructura esperada del conjunto de datos:

- **`(569, 30)`** — 569 muestras (casos) y 30 características numéricas, sin columnas
  irrelevantes.
- **Distribución de clases:**
  - Clase `0` (maligno): **212 casos** (~37%)
  - Clase `1` (benigno): **357 casos** (~63%)

Se observa un **desbalance moderado** a favor de la clase benigna. Esto justifica las
decisiones metodológicas posteriores: usar `stratify=y` al dividir los datos, elegir
**F1 macro** como métrica (en lugar de la exactitud) y aplicar **validación cruzada
estratificada**, para que el modelo no ignore la clase minoritaria (maligno), que es
justamente la de mayor interés clínico.

## Auditar antes de modelar

Antes de entrenar cualquier modelo, se realiza una **auditoría** del conjunto de
características (`X`) para conocer su estado y evitar decisiones automáticas mal
justificadas.

La tabla `audit` resume, por cada variable:

- **`tipo`** — tipo de dato de la columna.
- **`ausentes`** — cantidad de valores faltantes (NaN).
- **`unicos`** — número de valores distintos.

Además, se verifican dos condiciones clave:

- **Duplicados** — número de filas repetidas en `X`.
- **Target dentro de X** — confirma que la variable objetivo **no** esté entre las
  características, evitando así la fuga de información (*data leakage*).

**Criterios metodológicos:**

- ✅ Se **confirma la ausencia del target** dentro de `X`.
- 📊 Se **describe el balance** de las clases.
- ⚠️ **No se eliminan filas ni columnas** sin justificación explícita y documentada.

In [ ]:
audit = pd.DataFrame({
    "tipo": X.dtypes.astype(str),
    "ausentes": X.isna().sum(),
    "unicos": X.nunique()
})
print("Duplicados:", X.duplicated().sum())
print("Target dentro de X:", y.name in X.columns)
audit.head(10)

Duplicados: 0
Target dentro de X: False


,tipo,ausentes,unicos
mean radius,float64,0,456
mean texture,float64,0,479
mean perimeter,float64,0,522
mean area,float64,0,539
mean smoothness,float64,0,474
mean compactness,float64,0,537
mean concavity,float64,0,537
mean concave points,float64,0,542
mean symmetry,float64,0,432
mean fractal dimension,float64,0,499


### Comentario de la salida

La auditoría arroja un conjunto de datos limpio y listo para modelar:

- **Duplicados: 0** — no hay filas repetidas.
- **Target dentro de X: `False`** — la variable objetivo no está entre las
  características, por lo que **no hay riesgo de *data leakage***.
- **Ausentes: 0** en todas las columnas — no hay valores faltantes que imputar.
- **Tipo: `float64`** en todas las variables — todas son numéricas, adecuadas para la
  SVM sin necesidad de codificación.
- **Únicos** — cada variable presenta muchos valores distintos (400+), coherente con
  mediciones continuas.

En cuanto al **balance de clases**, se mantiene el desbalance moderado ya identificado
(357 benignos ~63% vs. 212 malignos ~37%).

Dado que no hay duplicados, ni valores faltantes, ni columnas no numéricas, **no existe
justificación para eliminar filas o columnas**: el criterio de no eliminar nada sin
razón se cumple de forma natural, y se preservan íntegros los datos.

## Reservar prueba

Antes de cualquier procesamiento o entrenamiento, se separa el conjunto de datos en
**entrenamiento** y **prueba**. Esto permite evaluar el modelo sobre datos que nunca
vio, obteniendo una estimación honesta de su capacidad de generalización.

La función `train_test_split` se configura con:

- **`test_size=0.20`** — reserva el 20% para prueba y el 80% para entrenamiento.
- **`random_state=42`** — fija la semilla para que la división sea **reproducible**.
- **`stratify=y`** — mantiene la **misma proporción de clases** en ambos conjuntos,
  clave en datos desbalanceados para que sean representativos.

> ⚠️ **Importante:** No se ajusta el `StandardScaler` (ni ningún transformador) **antes**
> de esta división. Hacerlo provocaría *data leakage*, pues el escalador aprendería
> estadísticas usando también los datos de prueba. El escalado se hará más adelante,
> **dentro del pipeline**, ajustándose solo con `X_train`.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))
print(y_test.value_counts(normalize=True).round(3))

(455, 30) (114, 30)
target
1    0.626
0    0.374
Name: proportion, dtype: float64
target
1    0.632
0    0.368
Name: proportion, dtype: float64


### Comentario de la salida

La división se realizó correctamente y la estratificación funcionó como se esperaba:

- **Tamaños:** entrenamiento `(455, 30)` y prueba `(114, 30)`, es decir, 80% y 20% de
  las 569 muestras, conservando las 30 características.
- **Proporción de clases:**

  | Clase | Entrenamiento | Prueba |
  |-------|---------------|--------|
  | 1 (benigno) | 62.6% | 63.2% |
  | 0 (maligno) | 37.4% | 36.8% |

Las proporciones son prácticamente idénticas entre ambos conjuntos y coinciden con el
balance original (~63% / ~37%). Esto confirma que **`stratify=y` preservó la
distribución de clases**, evitando que el conjunto de prueba quedara sesgado hacia una
clase. Con esta partición estratificada, la evaluación posterior será representativa
del problema real.

## Crear una línea base

Antes de entrenar un modelo real, se establece una **línea base** (*baseline*): un
modelo trivial que sirve como punto de referencia mínimo. Cualquier modelo posterior
debe superar claramente este umbral para justificar su complejidad.

Se utiliza un **`DummyClassifier`** que no aprende patrones reales de los datos:

- **`strategy="most_frequent"`** — siempre predice la clase más frecuente del
  entrenamiento (aquí, la clase benigna), ignorando por completo las características.

La evaluación usa **F1 macro**, que promedia el F1 de cada clase por separado dándoles
igual peso. Esto penaliza a un modelo que ignore la clase minoritaria, algo apropiado
en este dataset desbalanceado.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
print("F1 macro baseline:", f1_score(y_test, dummy_pred, average="macro"))

F1 macro baseline: 0.3870967741935484


### Comentario de la salida

El resultado es:

- **F1 macro baseline: `0.3871`**

Este valor bajo era el esperado: como el modelo *dummy* predice **siempre la clase
benigna (1)**, acierta en todos los casos benignos pero **falla en el 100% de los casos
malignos**. El F1 de la clase maligna es 0, y al promediarlo (macro) con el de la clase
benigna se obtiene ~0.39.

Este número es la **vara mínima a superar**: cualquier modelo con verdadera capacidad
predictiva —como la SVM que se entrena a continuación— debe alcanzar un F1 macro
sustancialmente mayor. Si un modelo sofisticado apenas igualara este 0.3871, sería
señal de que no está aprendiendo nada útil.

## Construir la SVM correctamente

Se construye un modelo de **Máquina de Vectores de Soporte** (*Support Vector Machine*,
SVM) encapsulado en un **`Pipeline`**, que encadena los pasos en el orden correcto y
evita la fuga de información (*data leakage*).

El pipeline tiene dos etapas:

- **`scale` (`StandardScaler`)** — estandariza las características (media 0, desviación 1).
  Es **imprescindible** en las SVM, sensibles a la escala. Al estar dentro del pipeline,
  el escalador se ajusta **solo con los datos de entrenamiento** de cada partición.
- **`model` (`SVC`)** — el clasificador SVM, con estos hiperparámetros:
  - **`kernel="rbf"`** — kernel de base radial, para fronteras no lineales.
  - **`C=1`** — regularización; equilibra margen amplio vs. errores de clasificación.
  - **`gamma="scale"`** — alcance de influencia de cada punto, ajustado según la varianza.
  - **`probability=True`** — habilita estimación de probabilidades (útil para ROC-AUC).
  - **`random_state=42`** — asegura la **reproducibilidad**.

`svm.fit(X_train, y_train)` entrena todo el pipeline: primero ajusta el escalador y
luego entrena la SVM sobre los datos ya estandarizados.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=42))
])
svm.fit(X_train, y_train)

c:\Users\Mudstart\Documents\Documentos\Maestria en Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scale', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['mean radius','mean texture','mean perimeter',...,'worst concave points', 'worst symmetry','worst fractal dimension']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


### Comentario de la salida

La salida muestra la representación del **`Pipeline` ya entrenado**, con sus dos pasos:
`StandardScaler` seguido de `SVC` con los hiperparámetros configurados. Esto confirma
que el ajuste (`fit`) se completó sin errores.

La clave metodológica —y la razón del título *"correctamente"*— es que el escalado
quedó **dentro** del pipeline y no aplicado por separado de antemano. Así, cuando se
haga validación cruzada o búsqueda de hiperparámetros, el `StandardScaler` se reajusta
en cada *fold* usando únicamente los datos de entrenamiento de esa partición,
respetando la separación entre entrenamiento y prueba. Todavía no hay métricas: este
paso solo deja el modelo listo para evaluarlo en el bloque siguiente.

## Evaluar la configuración base

Se evalúa el desempeño de la SVM sobre el **conjunto de prueba** (datos no vistos en el
entrenamiento), lo que da una estimación honesta de su generalización.

Se generan dos salidas del modelo:

- **`pred`** — las clases predichas (`0 = maligno`, `1 = benigno`).
- **`proba`** — la probabilidad estimada de la clase positiva, necesaria para el ROC-AUC.

Métricas y visualizaciones:

- **`classification_report`** — precisión, recall y F1 por clase (`digits=4` para más detalle).
- **`roc_auc_score`** — área bajo la curva ROC; mide la separación entre clases
  independientemente del umbral (1.0 = perfecto, 0.5 = azar).
- **`ConfusionMatrixDisplay`** — grafica la matriz de confusión, con aciertos y errores
  por clase.

> ⚠️ **Advertencia metodológica:** No se deben seleccionar hiperparámetros observando
> repetidamente esta matriz del conjunto de prueba. Hacerlo introduce *data leakage*
> indirecto: el conjunto de prueba dejaría de ser "no visto". La búsqueda de
> hiperparámetros se hará con validación cruzada sobre el entrenamiento.

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score)
import matplotlib.pyplot as plt

Path("../images").mkdir(exist_ok=True)
pred = svm.predict(X_test)
proba = svm.predict_proba(X_test)[:, 1]
print(classification_report(y_test, pred, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("SVM base - conjunto de prueba")
plt.tight_layout()
plt.savefig("../images/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'Path' is not defined

### Comentario de la salida

El modelo SVM base logra un desempeño excelente sobre el conjunto de prueba:

- **Accuracy: 0.9825** · **F1 macro: 0.9812** · **ROC-AUC: 0.995**
- Clase 0 (maligno): precision 0.9762, recall 0.9762
- Clase 1 (benigno): precision 0.9861, recall 0.9861

El F1 macro (0.9812) supera ampliamente la línea base (0.3871), confirmando que el
modelo sí aprende patrones útiles.

**Conteo de errores (matriz de confusión):**

|  | Predicho maligno (0) | Predicho benigno (1) |
|---|---|---|
| **Real maligno (0)** | 41 ✅ | **1** ❌ |
| **Real benigno (1)** | **1** ❌ | 71 ✅ |

Solo hay **2 errores** de 114 casos:

- **1 tumor maligno clasificado como benigno** (fila 0, columna 1).
- **1 tumor benigno clasificado como maligno** (fila 1, columna 0).

**¿Cuál error es más costoso?** El **maligno clasificado como benigno** es, con
diferencia, el más grave: significa decirle a un paciente que está sano cuando en
realidad tiene un tumor maligno, lo que **retrasa el tratamiento y pone en riesgo su
vida**. El error opuesto (benigno marcado como maligno) genera pruebas adicionales y
ansiedad, pero no compromete la vida. Por eso, en este dominio se prioriza **maximizar
el recall de la clase maligna** (minimizar ese falso negativo clínico), aunque cueste
algún falso positivo.

## Buscar C y gamma dentro del pipeline

Se realiza una **búsqueda de hiperparámetros** para encontrar la mejor combinación de
`C` y `gamma`, usando **validación cruzada** sobre el conjunto de entrenamiento. Así se
respeta la advertencia previa: el conjunto de prueba permanece intacto.

- **`StratifiedKFold`** — divide el entrenamiento en 5 particiones manteniendo la
  proporción de clases; `shuffle=True` con `random_state=42` lo hace aleatorio pero
  **reproducible**.
- **`GridSearchCV`** — prueba exhaustivamente todas las combinaciones de la rejilla,
  evaluando cada una por validación cruzada.

La rejilla (`param_grid`) explora `model__C` en `[0.1, 1, 10]` y `model__gamma` en
`["scale", 0.01, 0.1]`. El prefijo **`model__`** indica que esos parámetros pertenecen
al paso `model` (el `SVC`) **dentro del pipeline**: como la búsqueda opera sobre el
pipeline completo, el `StandardScaler` se reajusta en cada *fold* solo con sus datos de
entrenamiento, evitando *data leakage*.

Otros parámetros: **`scoring="f1_macro"`** (coherente con la línea base), **`n_jobs=-1`**
(paraleliza en todos los núcleos) y **`return_train_score=True`** (guarda scores de
entrenamiento, útil para detectar sobreajuste).

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    estimator=svm,
    param_grid={
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.01, 0.1]
    },
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True
)
search.fit(X_train, y_train)
print(search.best_params_)
print("CV:", round(search.best_score_, 4))

{'model__C': 10, 'model__gamma': 0.01}
CV: 0.9739


c:\Users\Mudstart\Documents\Documentos\Maestria en Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### Comentario de la salida

La búsqueda evaluó las 9 combinaciones (3 valores de `C` × 3 de `gamma`) mediante
validación cruzada de 5 folds, y encontró:

- **Mejores hiperparámetros:** `C = 10`, `gamma = 0.01`
- **Mejor F1 macro en CV:** `0.9739`

La combinación ganadora usa un `C` más alto (10) que el de la configuración base
(1), lo que significa menos regularización y una frontera de decisión que se ajusta un
poco más a los datos de entrenamiento; junto con un `gamma` bajo (0.01), que suaviza el
alcance de influencia de cada punto, logra el mejor equilibrio.

Este F1 macro de 0.9739 es un promedio **sobre datos de validación** (no de prueba), por
lo que es una estimación robusta y honesta del desempeño esperado. Nota que este valor
puede diferir ligeramente del que se obtenga luego en el conjunto de prueba, lo cual es
normal. En el siguiente bloque se examina la variación entre folds y se evalúa el mejor
modelo sobre la prueba.

## Comparar sin ocultar variación

Se examinan los resultados **completos** de la búsqueda, prestando atención no solo al
mejor promedio sino también a la **variación entre folds**. Esto evita elegir una
configuración que parezca buena "en promedio" pero sea inestable.

A partir de `search.cv_results_` se construye un DataFrame y se muestran las columnas
más informativas:

- **`param_model__C`** y **`param_model__gamma`** — la combinación evaluada.
- **`mean_test_score`** — F1 macro promedio en validación cruzada (rendimiento).
- **`std_test_score`** — desviación estándar entre folds; **mide la estabilidad**. Una
  desviación alta indica desempeño variable según la partición.
- **`mean_fit_time`** — tiempo promedio de entrenamiento (relevante para la eficiencia).

Reportar la desviación junto al promedio es una práctica de honestidad metodológica:
**no oculta la variación** que un solo número resumido escondería.

Finalmente, `search.best_estimator_` recupera el pipeline reentrenado con los mejores
hiperparámetros y se evalúa sobre el conjunto de **prueba** con el `classification_report`.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
columns = ["param_model__C", "param_model__gamma",
           "mean_test_score", "std_test_score", "mean_fit_time"]
cv_results[columns].sort_values("mean_test_score", ascending=False).head(9)

best = search.best_estimator_
best_pred = best.predict(X_test)
print(classification_report(y_test, best_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9762    0.9762    0.9762        42
           1     0.9861    0.9861    0.9861        72

    accuracy                         0.9825       114
   macro avg     0.9812    0.9812    0.9812       114
weighted avg     0.9825    0.9825    0.9825       114



### Comentario de la salida

**Tabla de validación cruzada (9 combinaciones, ordenadas por `mean_test_score`):**

- La mejor es `C=10, gamma=0.01` con **F1 macro 0.9739** y una desviación baja
  (**std ≈ 0.018**), lo que indica un desempeño **estable** entre folds.
- Las cuatro primeras combinaciones están muy próximas (0.967–0.974), con desviaciones
  similares (~0.014–0.018): varias configuraciones son competitivas, no hay una
  claramente superior por un margen enorme.
- Las peores (con `gamma=0.1` o `C=0.1`) bajan a ~0.934 y una, `C=10, gamma=0.1`,
  además muestra la **mayor variación (std ≈ 0.024)**: menos rendimiento y menos
  estable, la peor de ambos mundos.
- Los `mean_fit_time` son todos muy pequeños (~0.008–0.017 s), así que la eficiencia no
  es un factor diferenciador aquí.

La lección: mirar el promedio **y** la desviación juntos evita elegir a ciegas. La
configuración ganadora no solo tiene el mejor promedio, sino también una variación
contenida.

**Evaluación del mejor modelo en prueba:**

- **Accuracy: 0.9825** · **F1 macro: 0.9812**
- Clase 0 (maligno): precision 0.9762, recall 0.9762
- Clase 1 (benigno): precision 0.9861, recall 0.9861

Curiosamente, el mejor modelo optimizado obtiene en prueba el **mismo desempeño** que
la SVM base (0.9812 de F1 macro). Esto indica que el problema es relativamente sencillo
y que la configuración base ya era muy buena; la optimización confirma la robustez del
resultado más que producir un salto grande. Aun así, el modelo final generaliza
excelentemente sobre datos no vistos.

## Generación de reporte

Como paso final, se **persisten los artefactos** del experimento, garantizando la
**trazabilidad** y permitiendo reutilizar el modelo sin reentrenarlo.

- **`Path("../reports").mkdir(exist_ok=True)`** — crea la carpeta de salida si no existe
  (`exist_ok=True` evita error si ya está creada).
- **`cv_results[columns].to_csv("../reports/...")`** — guarda la tabla de resultados de
  validación cruzada en formato CSV.
- **`joblib.dump(best, "../reports/...")`** — serializa el mejor pipeline completo
  (escalador + SVM con sus hiperparámetros óptimos) en disco.

Las tres rutas usan `../reports/` para apuntar a la carpeta `reports/` de la raíz del
proyecto. Estas salidas dejan constancia reproducible
de qué modelo se eligió y con qué resultados, cerrando el ciclo del experimento.

In [ ]:
from pathlib import Path
import joblib
Path("../reports").mkdir(exist_ok=True)
cv_results[columns].to_csv("../reports/svm_cv_results.csv", index=False)
joblib.dump(best, "../reports/svm_best.joblib")

['../reports/svm_best.joblib']

### Comentario de la salida

La salida `['../reports/svm_best.joblib']` es el **valor de retorno de `joblib.dump`**,
que confirma la ruta del archivo del modelo guardado. La ejecución fue exitosa (no
imprime mensajes adicionales; solo crea los archivos en disco).

Quedan generados dos artefactos en la carpeta `reports/` de la raíz del proyecto 

- **`svm_cv_results.csv`** — la tabla con las 9 combinaciones de hiperparámetros y sus
  métricas de validación cruzada.
- **`svm_best.joblib`** — el pipeline entrenado (StandardScaler + SVM con `C=10`,
  `gamma=0.01`), listo para recargarse con `joblib.load()` y predecir sobre nuevos datos
  sin volver a entrenar.